In [1]:
# =====================================================
# Customer Behavior Drift Detection - Synthetic Data Generation
# =====================================================
# Generates realistic e-commerce data with behavioral drift

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration
OUTPUT_DIR = r"E:\data\E commerce data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Time parameters (18 months)
START_DATE = datetime(2024, 7, 1)
END_DATE = datetime(2025, 12, 31)
DRIFT_START_MONTH = 9  # Month 9 is when drift begins (March 2025)
SIGNUP_END_DATE = datetime(2024, 12, 31)  # Signups only in first 6 months

# Dataset sizes
NUM_CUSTOMERS = 7000
TARGET_EVENTS = 500000
TARGET_ORDERS = 120000

print(f"Output directory: {OUTPUT_DIR}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"Drift starts at month: {DRIFT_START_MONTH}")

Output directory: E:\data\E commerce data
Date range: 2024-07-01 to 2025-12-31
Drift starts at month: 9


In [2]:
# =====================================================
# TABLE 1: CUSTOMERS (7,000 rows)
# =====================================================

def generate_customers(n_customers):
    """Generate customer master data with realistic distributions."""
    
    customers = []
    
    # Country and region mapping
    country_regions = {
        'Egypt': ['Cairo', 'Giza', 'Alexandria', 'Delta'],
        'UAE': ['Gulf'],
        'Saudi': ['Gulf'],
        'Jordan': ['Gulf']
    }
    
    # Country distribution (Egypt dominant for regional e-commerce)
    countries = np.random.choice(
        ['Egypt', 'UAE', 'Saudi', 'Jordan'],
        size=n_customers,
        p=[0.55, 0.20, 0.18, 0.07]
    )
    
    # Acquisition channel distribution
    acquisition_channels = np.random.choice(
        ['organic', 'paid_ads', 'referral'],
        size=n_customers,
        p=[0.45, 0.35, 0.20]
    )
    
    # Device preference (mobile-heavy market)
    device_preferences = np.random.choice(
        ['mobile', 'web'],
        size=n_customers,
        p=[0.72, 0.28]
    )
    
    # Account type (70% free, 30% premium)
    account_types = np.random.choice(
        ['free', 'premium'],
        size=n_customers,
        p=[0.70, 0.30]
    )
    
    # Gender distribution
    genders = np.random.choice(
        ['male', 'female', 'other'],
        size=n_customers,
        p=[0.48, 0.50, 0.02]
    )
    
    for i in range(n_customers):
        customer_id = i + 1
        
        # Signup date within first 6 months (weighted towards earlier months)
        days_range = (SIGNUP_END_DATE - START_DATE).days
        # Use beta distribution to weight towards earlier signups
        signup_offset = int(np.random.beta(2, 3) * days_range)
        signup_date = START_DATE + timedelta(days=signup_offset)
        
        # Birth year (realistic age distribution for e-commerce: 18-65)
        birth_year = int(np.random.normal(1988, 10))
        birth_year = max(1960, min(2005, birth_year))
        
        country = countries[i]
        region = random.choice(country_regions[country])
        
        customers.append({
            'customer_id': customer_id,
            'signup_date': signup_date.date(),
            'birth_year': birth_year,
            'gender': genders[i],
            'country': country,
            'region': region,
            'acquisition_channel': acquisition_channels[i],
            'device_preference': device_preferences[i],
            'account_type': account_types[i]
        })
    
    return pd.DataFrame(customers)

# Generate customers
customers_df = generate_customers(NUM_CUSTOMERS)
print(f"Generated {len(customers_df)} customers")
print(f"\nCustomer Distribution Summary:")
print(f"  Countries: {customers_df['country'].value_counts().to_dict()}")
print(f"  Account Types: {customers_df['account_type'].value_counts().to_dict()}")
print(f"  Device Preference: {customers_df['device_preference'].value_counts().to_dict()}")
customers_df.head(10)

Generated 7000 customers

Customer Distribution Summary:
  Countries: {np.str_('Egypt'): 3878, np.str_('UAE'): 1383, np.str_('Saudi'): 1232, np.str_('Jordan'): 507}
  Account Types: {np.str_('free'): 4897, np.str_('premium'): 2103}
  Device Preference: {np.str_('mobile'): 4988, np.str_('web'): 2012}


,customer_id,signup_date,birth_year,gender,country,region,acquisition_channel,device_preference,account_type
0,1,2024-08-22,1980,female,Egypt,Cairo,organic,mobile,premium
1,2,2024-10-27,1994,male,Jordan,Gulf,organic,mobile,free
2,3,2024-07-24,1993,female,UAE,Gulf,referral,mobile,premium
3,4,2024-08-14,1976,female,UAE,Gulf,paid_ads,mobile,free
4,5,2024-10-11,1990,male,Egypt,Giza,organic,web,free
5,6,2024-08-30,1986,male,Egypt,Giza,organic,web,free
6,7,2024-07-06,1987,male,Egypt,Cairo,paid_ads,mobile,premium
7,8,2024-07-16,1993,female,Saudi,Gulf,organic,mobile,free
8,9,2024-10-15,1980,male,UAE,Gulf,organic,mobile,free
9,10,2024-08-25,1986,male,UAE,Gulf,organic,web,free


In [3]:
# =====================================================
# HELPER FUNCTIONS FOR DRIFT SIMULATION
# =====================================================

def get_month_number(date, start_date=START_DATE):
    """Calculate month number from start date (1-indexed)."""
    return (date.year - start_date.year) * 12 + (date.month - start_date.month) + 1

def get_drift_factor(month_num, drift_start=DRIFT_START_MONTH):
    """
    Calculate drift intensity factor.
    - Months 1-8: No drift (factor = 0)
    - Months 9-14: Gradual drift (factor = 0.1 to 0.8)
    - Months 15+: Full drift (factor = 1.0)
    """
    if month_num < drift_start:
        return 0.0
    elif month_num <= 14:
        # Gradual increase from 0.1 to 0.8
        return 0.1 + (month_num - drift_start) * 0.12
    else:
        return 1.0

def apply_seasonality(base_rate, date):
    """Apply seasonal factors to activity rates."""
    month = date.month
    # Higher activity during holiday seasons and sales periods
    seasonality = {
        1: 0.85,   # Post-holiday slowdown
        2: 0.90,
        3: 0.95,
        4: 1.00,
        5: 1.05,
        6: 1.10,   # Summer shopping
        7: 1.15,   # Summer peak
        8: 1.10,
        9: 1.00,   # Back to school
        10: 1.05,
        11: 1.25,  # Black Friday / Singles Day
        12: 1.30   # Holiday peak
    }
    return base_rate * seasonality.get(month, 1.0)

# Test drift factors
print("Drift Factor by Month:")
for m in range(1, 19):
    print(f"  Month {m}: {get_drift_factor(m):.2f}")

Drift Factor by Month:
  Month 1: 0.00
  Month 2: 0.00
  Month 3: 0.00
  Month 4: 0.00
  Month 5: 0.00
  Month 6: 0.00
  Month 7: 0.00
  Month 8: 0.00
  Month 9: 0.10
  Month 10: 0.22
  Month 11: 0.34
  Month 12: 0.46
  Month 13: 0.58
  Month 14: 0.70
  Month 15: 1.00
  Month 16: 1.00
  Month 17: 1.00
  Month 18: 1.00


In [4]:
# =====================================================
# TABLE 2: EVENTS (~500,000 rows)
# =====================================================

def generate_events(customers_df, target_events):
    """
    Generate event data with behavioral drift.
    Events follow logical flow: login → browse → add_to_cart → purchase
    """
    
    events = []
    event_id = 0
    purchase_events = []  # Track for order generation
    
    # App versions with release timeline
    app_versions = ['v1.0', 'v1.1', 'v1.2', 'v2.0', 'v2.1', 'v2.2', 'v3.0']
    
    page_categories = ['home', 'search', 'product', 'checkout']
    
    # Calculate events per customer (variable based on engagement)
    avg_events_per_customer = target_events / len(customers_df)
    
    for _, customer in customers_df.iterrows():
        customer_id = customer['customer_id']
        signup_date = pd.to_datetime(customer['signup_date'])
        device_pref = customer['device_preference']
        account_type = customer['account_type']
        country = customer['country']
        
        # Customer engagement level (premium users more active)
        if account_type == 'premium':
            engagement_multiplier = np.random.uniform(1.2, 1.6)
        else:
            engagement_multiplier = np.random.uniform(0.6, 1.2)
        
        # Mobile users have more frequent but shorter sessions
        if device_pref == 'mobile':
            session_frequency_boost = 1.15
        else:
            session_frequency_boost = 1.0
        
        # Number of events for this customer
        customer_events = int(avg_events_per_customer * engagement_multiplier * 
                             session_frequency_boost * np.random.uniform(0.5, 1.5))
        customer_events = max(10, customer_events)  # Minimum 10 events per customer
        
        # Generate dates for this customer's events
        available_days = (END_DATE - signup_date).days
        if available_days <= 0:
            continue
            
        # Generate session timestamps
        current_date = signup_date
        sessions_generated = 0
        
        while len([e for e in events if e['customer_id'] == customer_id]) < customer_events and current_date < END_DATE:
            # Determine activity probability for this day
            month_num = get_month_number(current_date.to_pydatetime())
            drift_factor = get_drift_factor(month_num)
            
            # Base daily activity probability
            base_prob = 0.15 if account_type == 'premium' else 0.08
            daily_prob = apply_seasonality(base_prob, current_date)
            
            if random.random() < daily_prob:
                # Generate a session
                session_id = f"sess_{customer_id}_{sessions_generated:05d}"
                sessions_generated += 1
                
                # Session start time (weighted towards evening hours)
                hour = int(np.random.normal(18, 5)) % 24
                minute = random.randint(0, 59)
                second = random.randint(0, 59)
                session_start = current_date.replace(hour=hour, minute=minute, second=second)
                
                # Determine device for this session
                if random.random() < 0.85:
                    device_type = device_pref
                else:
                    device_type = 'web' if device_pref == 'mobile' else 'mobile'
                
                # Base session duration (with drift for mobile)
                if device_type == 'mobile':
                    base_duration = np.random.normal(600, 300)  # ~10 min average
                    # Apply drift: reduce session duration after month 9
                    if drift_factor > 0:
                        base_duration *= (1 - 0.15 * drift_factor)
                else:
                    base_duration = np.random.normal(900, 400)  # ~15 min average
                
                session_duration = int(max(30, min(3600, base_duration)))
                
                # App version based on date
                days_since_start = (current_date - START_DATE).days
                version_idx = min(len(app_versions) - 1, days_since_start // 80)
                app_version = app_versions[version_idx]
                
                # Is logged in (higher for premium)
                is_logged_in = random.random() < (0.95 if account_type == 'premium' else 0.75)
                
                # Generate event sequence for this session
                # Determine session type based on drift
                if drift_factor > 0:
                    # After drift: more browsing, less purchasing
                    purchase_prob = max(0.08, 0.20 - 0.12 * drift_factor)
                    cart_prob = max(0.15, 0.30 - 0.10 * drift_factor)
                else:
                    purchase_prob = 0.20 if account_type == 'premium' else 0.15
                    cart_prob = 0.30
                
                # Event sequence for this session
                sequence = ['login', 'browse']
                
                # Add more browse events
                num_browses = random.randint(1, 5)
                if drift_factor > 0:
                    num_browses += random.randint(0, 3)  # More browsing during drift
                
                for _ in range(num_browses):
                    sequence.append('browse')
                
                # Possibly add to cart
                if random.random() < cart_prob:
                    sequence.append('add_to_cart')
                    # Possibly purchase
                    if random.random() < purchase_prob:
                        sequence.append('purchase')
                
                # Generate events from sequence
                event_time = session_start
                for event_type in sequence:
                    event_id += 1
                    
                    # Page category based on event type
                    if event_type == 'login':
                        page_cat = 'home'
                    elif event_type == 'browse':
                        page_cat = random.choice(['home', 'search', 'product'])
                    elif event_type == 'add_to_cart':
                        page_cat = 'product'
                    else:  # purchase
                        page_cat = 'checkout'
                    
                    event_record = {
                        'event_id': event_id,
                        'customer_id': customer_id,
                        'event_timestamp': event_time,
                        'event_date': event_time.date(),
                        'event_type': event_type,
                        'session_id': session_id,
                        'session_duration_sec': session_duration,
                        'page_category': page_cat,
                        'device_type': device_type,
                        'app_version': app_version,
                        'is_logged_in': is_logged_in,
                        'geo_country': country
                    }
                    events.append(event_record)
                    
                    # Track purchase events for order generation
                    if event_type == 'purchase':
                        purchase_events.append({
                            'customer_id': customer_id,
                            'event_timestamp': event_time,
                            'account_type': account_type,
                            'region': customer['region'],
                            'month_num': month_num,
                            'drift_factor': drift_factor
                        })
                    
                    # Increment time within session
                    event_time += timedelta(seconds=random.randint(10, 180))
            
            # Move to next day
            current_date += timedelta(days=1)
    
    return pd.DataFrame(events), purchase_events

print("Generating events (this may take a moment)...")
events_df, purchase_events = generate_events(customers_df, TARGET_EVENTS)
print(f"\nGenerated {len(events_df)} events")
print(f"Purchase events available for orders: {len(purchase_events)}")
print(f"\nEvent Type Distribution:")
print(events_df['event_type'].value_counts())

Generating events (this may take a moment)...

Generated 598321 events
Purchase events available for orders: 5547

Event Type Distribution:
event_type
browse         452084
login          108434
add_to_cart     32256
purchase         5547
Name: count, dtype: int64


In [5]:
# =====================================================
# TABLE 3: ORDERS (~120,000 rows)
# =====================================================

def generate_orders(purchase_events, target_orders):
    """
    Generate order data linked to purchase events.
    Includes payment failures, refunds, and drift effects.
    """
    
    orders = []
    product_categories = ['electronics', 'fashion', 'groceries']
    payment_methods = ['card', 'wallet', 'cash']
    
    # Sample purchase events if we have more than target
    if len(purchase_events) > target_orders:
        sampled_purchases = random.sample(purchase_events, target_orders)
    else:
        # Need to generate additional orders by repeating some customers
        sampled_purchases = purchase_events.copy()
        additional_needed = target_orders - len(purchase_events)
        sampled_purchases.extend(random.choices(purchase_events, k=additional_needed))
    
    for i, purchase in enumerate(sampled_purchases):
        order_id = i + 1
        customer_id = purchase['customer_id']
        order_timestamp = purchase['event_timestamp']
        account_type = purchase['account_type']
        region = purchase['region']
        drift_factor = purchase['drift_factor']
        
        # Order value based on account type and drift
        if account_type == 'premium':
            base_order_value = np.random.lognormal(7.0, 0.5)  # Higher average
            base_order_value = max(400, min(2000, base_order_value))
        else:
            base_order_value = np.random.lognormal(6.5, 0.6)
            base_order_value = max(400, min(2000, base_order_value))
        
        # Apply drift: order value decreases slightly
        if drift_factor > 0:
            base_order_value *= (1 - 0.08 * drift_factor)
        
        order_value = round(max(400, min(2000, base_order_value)), 2)
        
        # Items count (correlated with order value)
        items_count = max(1, min(10, int(order_value / 200) + random.randint(-1, 2)))
        
        # Payment method distribution
        payment_method = np.random.choice(
            payment_methods,
            p=[0.50, 0.35, 0.15]
        )
        
        # Payment status with drift effects
        # Base rates: failed ~7-10%, refunded ~3-5%
        base_fail_rate = 0.08
        base_refund_rate = 0.04
        
        # Drift increases failure rate
        if drift_factor > 0:
            fail_rate = base_fail_rate + 0.04 * drift_factor
            refund_rate = base_refund_rate + 0.02 * drift_factor
        else:
            fail_rate = base_fail_rate
            refund_rate = base_refund_rate
        
        # Premium users have lower refund rate
        if account_type == 'premium':
            refund_rate *= 0.6
        
        status_roll = random.random()
        if status_roll < fail_rate:
            payment_status = 'failed'
        elif status_roll < fail_rate + refund_rate:
            payment_status = 'refunded'
        else:
            payment_status = 'success'
        
        # Discount applied (increases with drift)
        base_discount_prob = 0.25
        if drift_factor > 0:
            discount_prob = min(0.60, base_discount_prob + 0.30 * drift_factor)
        else:
            discount_prob = base_discount_prob
        
        discount_applied = random.random() < discount_prob
        
        if discount_applied:
            # Discount amount: 5-50% of order value
            discount_pct = np.random.beta(2, 5) * 0.45 + 0.05  # Weighted towards lower discounts
            # Higher discounts during drift period
            if drift_factor > 0:
                discount_pct *= (1 + 0.3 * drift_factor)
            discount_pct = min(0.50, discount_pct)
            discount_amount = round(order_value * discount_pct, 2)
        else:
            discount_amount = 0.0
        
        # Product category (some correlation with order value)
        if order_value > 1200:
            category_probs = [0.55, 0.30, 0.15]  # More electronics for high value
        elif order_value > 700:
            category_probs = [0.35, 0.45, 0.20]  # More fashion for mid value
        else:
            category_probs = [0.20, 0.35, 0.45]  # More groceries for low value
        
        product_category = np.random.choice(product_categories, p=category_probs)
        
        orders.append({
            'order_id': order_id,
            'customer_id': customer_id,
            'order_timestamp': order_timestamp,
            'order_date': order_timestamp.date() if hasattr(order_timestamp, 'date') else order_timestamp,
            'order_value': order_value,
            'currency': 'EGP',
            'items_count': items_count,
            'payment_method': payment_method,
            'payment_status': payment_status,
            'discount_applied': discount_applied,
            'discount_amount': discount_amount,
            'product_category': product_category,
            'shipping_region': region
        })
    
    return pd.DataFrame(orders)

print("Generating orders...")
orders_df = generate_orders(purchase_events, TARGET_ORDERS)
print(f"Generated {len(orders_df)} orders")
print(f"\nPayment Status Distribution:")
print(orders_df['payment_status'].value_counts(normalize=True).round(3))
print(f"\nProduct Category Distribution:")
print(orders_df['product_category'].value_counts())

Generating orders...
Generated 120000 orders

Payment Status Distribution:
payment_status
success     0.885
failed      0.082
refunded    0.033
Name: proportion, dtype: float64

Product Category Distribution:
product_category
fashion        44508
electronics    42082
groceries      33410
Name: count, dtype: int64


In [6]:
# =====================================================
# DATA VALIDATION & DRIFT VERIFICATION
# =====================================================

print("=" * 60)
print("DATA VALIDATION")
print("=" * 60)

# Check row counts
print(f"\n📊 Row Counts:")
print(f"  Customers: {len(customers_df):,} (target: {NUM_CUSTOMERS:,})")
print(f"  Events: {len(events_df):,} (target: ~{TARGET_EVENTS:,})")
print(f"  Orders: {len(orders_df):,} (target: ~{TARGET_ORDERS:,})")

# Check referential integrity
print(f"\n🔗 Referential Integrity:")
events_customer_ids = set(events_df['customer_id'].unique())
orders_customer_ids = set(orders_df['customer_id'].unique())
customer_ids = set(customers_df['customer_id'].unique())

print(f"  All event customer_ids exist in customers: {events_customer_ids.issubset(customer_ids)}")
print(f"  All order customer_ids exist in customers: {orders_customer_ids.issubset(customer_ids)}")

# Verify drift patterns
print(f"\n📈 Behavioral Drift Verification:")

# Add month column for analysis
events_df['month'] = pd.to_datetime(events_df['event_timestamp']).dt.to_period('M')
orders_df['month'] = pd.to_datetime(orders_df['order_timestamp']).dt.to_period('M')

# Session duration by month (for mobile users)
mobile_events = events_df[events_df['device_type'] == 'mobile'].copy()
mobile_session_duration = mobile_events.groupby('month')['session_duration_sec'].mean()

print(f"\n  Mobile Session Duration (avg seconds):")
for period in mobile_session_duration.index[-6:]:
    print(f"    {period}: {mobile_session_duration[period]:.0f}")

# Order value by month
order_value_by_month = orders_df.groupby('month')['order_value'].mean()
print(f"\n  Average Order Value by Month (last 6 months):")
for period in order_value_by_month.index[-6:]:
    print(f"    {period}: {order_value_by_month[period]:.2f} EGP")

# Discount usage by month
discount_by_month = orders_df.groupby('month')['discount_applied'].mean() * 100
print(f"\n  Discount Usage Rate by Month (last 6 months):")
for period in discount_by_month.index[-6:]:
    print(f"    {period}: {discount_by_month[period]:.1f}%")

# Payment failure rate by month
orders_df['is_failed'] = orders_df['payment_status'] == 'failed'
failure_by_month = orders_df.groupby('month')['is_failed'].mean() * 100
print(f"\n  Payment Failure Rate by Month (last 6 months):")
for period in failure_by_month.index[-6:]:
    print(f"    {period}: {failure_by_month[period]:.1f}%")

# Clean up temporary columns
events_df.drop('month', axis=1, inplace=True)
orders_df.drop(['month', 'is_failed'], axis=1, inplace=True)

DATA VALIDATION

📊 Row Counts:
  Customers: 7,000 (target: 7,000)
  Events: 598,321 (target: ~500,000)
  Orders: 120,000 (target: ~120,000)

🔗 Referential Integrity:
  All event customer_ids exist in customers: True
  All order customer_ids exist in customers: True

📈 Behavioral Drift Verification:

  Mobile Session Duration (avg seconds):
    2025-07: 561
    2025-08: 550
    2025-09: 539
    2025-10: 478
    2025-11: 560
    2025-12: 529

  Average Order Value by Month (last 6 months):
    2025-04: 858.37 EGP
    2025-05: 814.16 EGP
    2025-06: 827.21 EGP
    2025-07: 742.28 EGP
    2025-08: 712.20 EGP
    2025-10: 846.54 EGP

  Discount Usage Rate by Month (last 6 months):
    2025-04: 31.4%
    2025-05: 36.0%
    2025-06: 35.6%
    2025-07: 43.1%
    2025-08: 53.7%
    2025-10: 48.0%

  Payment Failure Rate by Month (last 6 months):
    2025-04: 7.9%
    2025-05: 9.0%
    2025-06: 8.1%
    2025-07: 9.3%
    2025-08: 9.8%
    2025-10: 24.0%


In [7]:
# =====================================================
# SAVE TO CSV FILES
# =====================================================

# Prepare final dataframes (ensure correct data types)
customers_final = customers_df.copy()
customers_final['signup_date'] = pd.to_datetime(customers_final['signup_date'])

events_final = events_df.copy()
events_final['event_timestamp'] = pd.to_datetime(events_final['event_timestamp'])
events_final['event_date'] = pd.to_datetime(events_final['event_date'])

orders_final = orders_df.copy()
orders_final['order_timestamp'] = pd.to_datetime(orders_final['order_timestamp'])
orders_final['order_date'] = pd.to_datetime(orders_final['order_date'])

# Save to CSV
customers_path = os.path.join(OUTPUT_DIR, 'customers.csv')
events_path = os.path.join(OUTPUT_DIR, 'events.csv')
orders_path = os.path.join(OUTPUT_DIR, 'orders.csv')

customers_final.to_csv(customers_path, index=False)
events_final.to_csv(events_path, index=False)
orders_final.to_csv(orders_path, index=False)

print("=" * 60)
print("FILES SAVED SUCCESSFULLY")
print("=" * 60)
print(f"\n📁 Output Directory: {OUTPUT_DIR}")
print(f"\n✅ customers.csv: {len(customers_final):,} rows")
print(f"✅ events.csv: {len(events_final):,} rows")
print(f"✅ orders.csv: {len(orders_final):,} rows")

# Display file sizes
import os
for filename in ['customers.csv', 'events.csv', 'orders.csv']:
    filepath = os.path.join(OUTPUT_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   {filename}: {size_mb:.2f} MB")

FILES SAVED SUCCESSFULLY

📁 Output Directory: E:\data\E commerce data

✅ customers.csv: 7,000 rows
✅ events.csv: 598,321 rows
✅ orders.csv: 120,000 rows
   customers.csv: 0.40 MB
   events.csv: 56.53 MB
   orders.csv: 11.01 MB


In [8]:
# =====================================================
# FINAL DATA PREVIEW
# =====================================================

print("=" * 60)
print("CUSTOMERS TABLE PREVIEW")
print("=" * 60)
display(customers_final.head(10))
print(f"\nColumns: {list(customers_final.columns)}")
print(f"Data Types:\n{customers_final.dtypes}")

print("\n" + "=" * 60)
print("EVENTS TABLE PREVIEW")
print("=" * 60)
display(events_final.head(10))
print(f"\nColumns: {list(events_final.columns)}")

print("\n" + "=" * 60)
print("ORDERS TABLE PREVIEW")
print("=" * 60)
display(orders_final.head(10))
print(f"\nColumns: {list(orders_final.columns)}")

print("\n" + "=" * 60)
print("DATASET GENERATION COMPLETE!")
print("=" * 60)
print("""
📊 Dataset Summary:
   - 18 months of e-commerce data (Jul 2024 - Dec 2025)
   - Behavioral drift begins at Month 9 (Mar 2025)
   - Clear drift detectable by Month 15 (Sep 2025)
   
🔍 Drift Indicators to Analyze:
   1. Session duration decreases for mobile users
   2. Purchase frequency drops while browsing increases
   3. Average order value decreases
   4. Discount dependency increases
   5. Payment failure rate increases slightly
   
📁 Files saved to: {output_dir}
   - customers.csv ({n_customers:,} rows)
   - events.csv ({n_events:,} rows)  
   - orders.csv ({n_orders:,} rows)
""".format(
    output_dir=OUTPUT_DIR,
    n_customers=len(customers_final),
    n_events=len(events_final),
    n_orders=len(orders_final)
))

CUSTOMERS TABLE PREVIEW


,customer_id,signup_date,birth_year,gender,country,region,acquisition_channel,device_preference,account_type
0,1,2024-08-22,1980,female,Egypt,Cairo,organic,mobile,premium
1,2,2024-10-27,1994,male,Jordan,Gulf,organic,mobile,free
2,3,2024-07-24,1993,female,UAE,Gulf,referral,mobile,premium
3,4,2024-08-14,1976,female,UAE,Gulf,paid_ads,mobile,free
4,5,2024-10-11,1990,male,Egypt,Giza,organic,web,free
5,6,2024-08-30,1986,male,Egypt,Giza,organic,web,free
6,7,2024-07-06,1987,male,Egypt,Cairo,paid_ads,mobile,premium
7,8,2024-07-16,1993,female,Saudi,Gulf,organic,mobile,free
8,9,2024-10-15,1980,male,UAE,Gulf,organic,mobile,free
9,10,2024-08-25,1986,male,UAE,Gulf,organic,web,free



Columns: ['customer_id', 'signup_date', 'birth_year', 'gender', 'country', 'region', 'acquisition_channel', 'device_preference', 'account_type']
Data Types:
customer_id                     int64
signup_date            datetime64[ns]
birth_year                      int64
gender                         object
country                        object
region                         object
acquisition_channel            object
device_preference              object
account_type                   object
dtype: object

EVENTS TABLE PREVIEW


,event_id,customer_id,event_timestamp,event_date,event_type,session_id,session_duration_sec,page_category,device_type,app_version,is_logged_in,geo_country
0,1,1,2024-08-25 17:37:26,2024-08-25,login,sess_1_00000,389,home,mobile,v1.0,True,Egypt
1,2,1,2024-08-25 17:39:41,2024-08-25,browse,sess_1_00000,389,home,mobile,v1.0,True,Egypt
2,3,1,2024-08-25 17:40:22,2024-08-25,browse,sess_1_00000,389,product,mobile,v1.0,True,Egypt
3,4,1,2024-08-25 17:42:01,2024-08-25,browse,sess_1_00000,389,home,mobile,v1.0,True,Egypt
4,5,1,2024-08-25 17:42:22,2024-08-25,browse,sess_1_00000,389,search,mobile,v1.0,True,Egypt
5,6,1,2024-08-25 17:44:35,2024-08-25,add_to_cart,sess_1_00000,389,product,mobile,v1.0,True,Egypt
6,7,1,2024-09-01 13:08:32,2024-09-01,login,sess_1_00001,389,home,mobile,v1.0,True,Egypt
7,8,1,2024-09-01 13:09:04,2024-09-01,browse,sess_1_00001,389,product,mobile,v1.0,True,Egypt
8,9,1,2024-09-01 13:12:04,2024-09-01,browse,sess_1_00001,389,product,mobile,v1.0,True,Egypt
9,10,1,2024-09-01 13:13:58,2024-09-01,browse,sess_1_00001,389,search,mobile,v1.0,True,Egypt



Columns: ['event_id', 'customer_id', 'event_timestamp', 'event_date', 'event_type', 'session_id', 'session_duration_sec', 'page_category', 'device_type', 'app_version', 'is_logged_in', 'geo_country']

ORDERS TABLE PREVIEW


,order_id,customer_id,order_timestamp,order_date,order_value,currency,items_count,payment_method,payment_status,discount_applied,discount_amount,product_category,shipping_region
0,1,1,2024-09-22 12:32:19,2024-09-22,621.02,EGP,5,cash,success,False,0.00,fashion,Cairo
1,2,3,2024-07-29 13:49:29,2024-07-29,461.76,EGP,1,wallet,success,True,51.49,groceries,Gulf
2,3,3,2024-08-26 17:30:42,2024-08-26,1982.08,EGP,10,wallet,success,False,0.00,fashion,Gulf
3,4,4,2024-12-16 11:05:05,2024-12-16,658.05,EGP,4,wallet,success,False,0.00,groceries,Gulf
4,5,5,2025-02-18 14:22:09,2025-02-18,569.47,EGP,2,wallet,success,False,0.00,groceries,Giza
5,6,6,2024-12-02 18:43:33,2024-12-02,2000.00,EGP,10,card,success,False,0.00,electronics,Giza
6,7,7,2024-09-09 04:38:38,2024-09-09,910.70,EGP,6,wallet,success,False,0.00,fashion,Cairo
7,8,7,2024-10-28 14:37:44,2024-10-28,2000.00,EGP,10,cash,success,True,205.64,electronics,Cairo
8,9,13,2024-07-26 21:18:26,2024-07-26,471.96,EGP,4,wallet,success,False,0.00,groceries,Gulf
9,10,13,2024-08-29 17:36:00,2024-08-29,660.93,EGP,3,card,success,False,0.00,fashion,Gulf



Columns: ['order_id', 'customer_id', 'order_timestamp', 'order_date', 'order_value', 'currency', 'items_count', 'payment_method', 'payment_status', 'discount_applied', 'discount_amount', 'product_category', 'shipping_region']

DATASET GENERATION COMPLETE!

📊 Dataset Summary:
   - 18 months of e-commerce data (Jul 2024 - Dec 2025)
   - Behavioral drift begins at Month 9 (Mar 2025)
   - Clear drift detectable by Month 15 (Sep 2025)

🔍 Drift Indicators to Analyze:
   1. Session duration decreases for mobile users
   2. Purchase frequency drops while browsing increases
   3. Average order value decreases
   4. Discount dependency increases
   5. Payment failure rate increases slightly

📁 Files saved to: E:\data\E commerce data
   - customers.csv (7,000 rows)
   - events.csv (598,321 rows)  
   - orders.csv (120,000 rows)

